# Multi-Seed N=12 Experiment

**Goal:** Run N=12 w=50 baseline with 8 different random seeds to test whether
the partial-basin failure pattern is stochastic (different solitons fail per seed)
or structural (same solitons always fail).

**Hypothesis A (stochastic):** Different seeds → different per-soliton failure patterns.
Implies loss landscape has multiple partial-basin attractors; optimizer falls in stochastically.

**Hypothesis B (structural):** All seeds → same solitons fail (soliton 2 always worst).
Implies the failure is config-determined, not optimization-stochastic.

**Config:** Canonical real config — same as all rerun notebooks.
- N=12, width=50, 15k Adam + 2k L-BFGS
- Seeds: [1, 7, 23, 42, 99, 137, 256, 999]
- Seed 99 is our standard baseline — should reproduce ~42% final L2

**Expected runtime:** ~30 min/seed x 8 seeds = ~4 hrs on T4x2 (runs sequentially)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import json
import os
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device = {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# CANONICAL CONFIG  (do not change)
# ============================================================

N          = 12
WIDTH      = 50
ADAM_ITER  = 15000
LBFGS_ITER = 2000      # 100 steps x max_iter=20
ADAM_LR    = 1e-3

SEEDS = [1, 7, 23, 42, 99, 137, 256, 999]

def make_config(N):
    """Canonical speed/x0/domain config. Matches all rerun notebooks."""
    speeds = np.linspace(1.5 + 0.5*(N-1), 1.5, N)  # fastest first
    x0s    = np.linspace((N-1)*5, -(N-1)*5, N)       # rightmost first
    c_max  = speeds[0]
    x_min  = x0s[-1] - 10
    x_max  = x0s[0]  + c_max * 5 + 15   # +15 matches canonical rerun domain
    domain = (round(x_min, 1), round(x_max, 1))
    return speeds, x0s, domain

speeds, x0s, domain = make_config(N)
x_lo, x_hi = domain
T           = 5.0
width       = x_hi - x_lo

# Point counts — canonical scaling from rerun notebooks
# N=12 baseline: PDE=56666, Data=1133, IC=1416, BC=200, width=170
# => 56666/170=333.3/unit, 1133/170=6.67/unit, 1416/170=8.33/unit
n_pde  = int(width * 333.33)
n_data = int(width * 6.67)
n_ic   = int(width * 8.33)
n_bc   = 200

print(f'N={N}  domain=({x_lo}, {x_hi})  width={width}')
print(f'speeds={list(np.round(speeds, 2))}')
print(f'x0s   ={list(x0s)}')
print(f'points: PDE={n_pde:,}  Data={n_data:,}  IC={n_ic:,}  BC={n_bc}')
print(f'seeds : {SEEDS}')

In [ ]:
# ============================================================
# PHYSICS  (positive-convention KdV)
# u_t + 6u*u_x + u_xxx = 0
# soliton: u = (c/2) * sech^2( (sqrt(c)/2) * (x - c*t - x0) )
# ============================================================

def soliton_np(x, t, c, x0):
    xi = (np.sqrt(c) / 2.0) * (x - c * t - x0)
    return (c / 2.0) / np.cosh(xi)**2

def soliton_th(x, t, c, x0):
    xi = (c**0.5 / 2.0) * (x - c * t - x0)
    return (c / 2.0) / torch.cosh(xi)**2

def exact_np(x, t, speeds, x0s):
    u = np.zeros_like(x)
    for c, x0 in zip(speeds, x0s):
        u += soliton_np(x, t, c, x0)
    return u

def exact_th(x, t, speeds, x0s):
    u = torch.zeros_like(x)
    for c, x0 in zip(speeds, x0s):
        u += soliton_th(x, t, c, x0)
    return u

In [ ]:
# ============================================================
# NETWORK  (6 hidden layers x WIDTH neurons, tanh, Xavier)
# ============================================================

class PINN(nn.Module):
    def __init__(self, width=50):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(5):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)
        self._xavier_init()

    def _xavier_init(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

def count_params(model):
    return sum(p.numel() for p in model.parameters())

_tmp = PINN(WIDTH)
print(f'width={WIDTH}  params={count_params(_tmp):,}')
del _tmp

In [ ]:
# ============================================================
# DATA BUILDER
# ============================================================

def build_data(x_lo, x_hi, T, n_pde, n_data, n_ic, n_bc,
               speeds, x0s, seed, device):
    rng = np.random.default_rng(seed)

    # PDE collocation
    xr = rng.uniform(x_lo, x_hi, n_pde).astype(np.float32)
    tr = rng.uniform(0.0,   T,    n_pde).astype(np.float32)

    # Scattered data
    xd = rng.uniform(x_lo, x_hi, n_data).astype(np.float32)
    td = rng.uniform(0.0,   T,    n_data).astype(np.float32)
    ud = exact_np(xd, td, speeds, x0s).astype(np.float32)

    # IC
    xi = np.linspace(x_lo, x_hi, n_ic).astype(np.float32)
    ti = np.zeros(n_ic, dtype=np.float32)
    ui = exact_np(xi, ti, speeds, x0s).astype(np.float32)

    # BC
    t_bcl = rng.uniform(0.0, T, n_bc // 2).astype(np.float32)
    t_bcr = rng.uniform(0.0, T, n_bc // 2).astype(np.float32)
    x_bcl = np.full(n_bc // 2, x_lo, dtype=np.float32)
    x_bcr = np.full(n_bc // 2, x_hi, dtype=np.float32)
    u_bcl = exact_np(x_bcl, t_bcl, speeds, x0s).astype(np.float32)
    u_bcr = exact_np(x_bcr, t_bcr, speeds, x0s).astype(np.float32)

    def tt(a):
        return torch.tensor(a, dtype=torch.float32, device=device).unsqueeze(1)

    data = {
        'xr': tt(xr), 'tr': tt(tr),
        'xd': tt(xd), 'td': tt(td), 'ud': tt(ud),
        'xi': tt(xi), 'ti': tt(ti), 'ui': tt(ui),
        'x_bcl': tt(x_bcl), 't_bcl': tt(t_bcl), 'u_bcl': tt(u_bcl),
        'x_bcr': tt(x_bcr), 't_bcr': tt(t_bcr), 'u_bcr': tt(u_bcr),
    }

    # hard assert — catches sign convention bugs and label corruption
    assert ud.max() > 0.01, f'data labels wrong: max={ud.max():.4f}'
    assert ui.max() > 0.01, f'IC labels wrong: max={ui.max():.4f}'

    return data

In [ ]:
# ============================================================
# LOSS
# ============================================================

def pde_residual(model, xr, tr):
    """KdV: u_t + 6*u*u_x + u_xxx = 0  (positive convention)"""
    xr = xr.requires_grad_(True)
    tr = tr.requires_grad_(True)
    u   = model(xr, tr)
    u_t  = torch.autograd.grad(u,   tr, torch.ones_like(u),   create_graph=True)[0]
    u_x  = torch.autograd.grad(u,   xr, torch.ones_like(u),   create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, xr, torch.ones_like(u_x), create_graph=True)[0]
    u_xxx= torch.autograd.grad(u_xx,xr, torch.ones_like(u_xx),create_graph=True)[0]
    return u_t + 6.0 * u * u_x + u_xxx

def compute_losses(model, data):
    res   = pde_residual(model, data['xr'], data['tr'])
    l_pde = torch.mean(res**2)

    u_ic  = model(data['xi'], data['ti'])
    l_ic  = torch.mean((u_ic - data['ui'])**2)

    u_bcl = model(data['x_bcl'], data['t_bcl'])
    u_bcr = model(data['x_bcr'], data['t_bcr'])
    l_bc  = (torch.mean((u_bcl - data['u_bcl'])**2) +
             torch.mean((u_bcr - data['u_bcr'])**2)) / 2.0

    u_d   = model(data['xd'], data['td'])
    l_data= torch.mean((u_d - data['ud'])**2)

    total = l_pde + l_ic + l_bc + l_data
    return total, l_pde, l_ic, l_bc, l_data

In [ ]:
# ============================================================
# L2 ERROR
# ============================================================

@torch.no_grad()
def compute_l2(model, x_lo, x_hi, T, speeds, x0s, nx=300, nt=100):
    xs = np.linspace(x_lo, x_hi, nx, dtype=np.float32)
    ts = np.linspace(0.0,   T,   nt, dtype=np.float32)
    Xg, Tg = np.meshgrid(xs, ts)
    xf = Xg.ravel(); tf = Tg.ravel()
    u_ex = exact_np(xf, tf, speeds, x0s)
    xt   = torch.tensor(np.stack([xf, tf], axis=1),
                        dtype=torch.float32, device=device)
    u_pr = model(xt[:, :1], xt[:, 1:]).cpu().numpy().ravel()
    num  = np.sqrt(np.mean((u_pr - u_ex)**2))
    den  = np.sqrt(np.mean(u_ex**2))
    return float((num / den) * 100.0)   # cast to Python float for JSON


In [ ]:
# ============================================================
# PER-SOLITON L2
# ============================================================

@torch.no_grad()
def per_soliton_l2(model, speeds, x0s, T, device, window=3.0, nt=100, nx_per_unit=20):
    """
    For each soliton, compute L2 error in a window of +/-`window` units
    around the soliton centre at each timestep, then average over time.
    """
    ts = np.linspace(0.0, T, nt, dtype=np.float32)
    results = []
    for c, x0 in zip(speeds, x0s):
        errs = []
        for t_val in ts:
            centre = c * t_val + x0
            xs = np.linspace(centre - window, centre + window,
                             max(10, int(2*window*nx_per_unit)),
                             dtype=np.float32)
            tf = np.full_like(xs, t_val)
            u_ex = exact_np(xs, tf, speeds, x0s)
            xt   = torch.tensor(np.stack([xs, tf], axis=1),
                                dtype=torch.float32, device=device)
            u_pr = model(xt[:, :1], xt[:, 1:]).cpu().numpy().ravel()
            num  = np.sqrt(np.mean((u_pr - u_ex)**2))
            den  = np.sqrt(np.mean(u_ex**2)) + 1e-10
            errs.append(num / den)
        results.append({
            'c':           float(c),
            'x0':          float(x0),
            'local_l2_pct': float(np.mean(errs)) * 100.0
        })
    return results

In [ ]:
# ============================================================
# PIPELINE SANITY CHECK  (N=3, 500 Adam, no L-BFGS)
# Reference from rerun notebooks: L2 after 500 Adam iters ~5.58%
# ============================================================

print('=' * 60)
print('PIPELINE SANITY CHECK  (N=3, short Adam, no L-BFGS)')
print('=' * 60)

_seed = 99
torch.manual_seed(_seed)
np.random.seed(_seed)

_sp, _x0, _dom = make_config(3)
_xl, _xh = _dom
_w = _xh - _xl
_m = PINN(50).to(device)
_d = build_data(_xl, _xh, 5.0,
                int(_w*333.33), int(_w*6.67), int(_w*8.33), 200,
                _sp, _x0, _seed, device)

print(f'  device = {device}')
print(f'  domain = ({_xl}, {_xh}) | n_pde={int(_w*333.33)} | n_data={int(_w*6.67)}')
print(f'  params = {count_params(_m):,}')

_opt = torch.optim.Adam(_m.parameters(), lr=1e-3)
_t0  = time.time()
for _i in range(500):
    _opt.zero_grad()
    _loss, _lp, _li, _lb, _ld = compute_losses(_m, _d)
    _loss.backward()
    _opt.step()
    if _i % 100 == 0:
        print(f'  Adam  {_i:4d} | Total={_loss.item():.6e} '
              f'PDE={_lp.item():.2e} IC={_li.item():.2e} '
              f'BC={_lb.item():.2e} Data={_ld.item():.2e}')

_l2 = compute_l2(_m, _xl, _xh, 5.0, _sp, _x0)
print(f'  >> Adam done {time.time()-_t0:.1f}s | L2={_l2:.4f}%')

_lbfgs = torch.optim.LBFGS(_m.parameters(), lr=1.0, max_iter=20,
                             history_size=50, tolerance_change=1e-12,
                             line_search_fn='strong_wolfe')
def _cl():
    _lbfgs.zero_grad()
    l, *_ = compute_losses(_m, _d)
    l.backward()
    return l
_sanity_loss = _lbfgs.step(_cl)
print(f'  LBFGS    0 | Total={_sanity_loss.item():.8e}')
print(f'  >> LBFGS done 0.1s | L2={_l2:.4f}%')
print(f'  L2 after 500 Adam iters: {_l2:.4f}%')
assert _l2 < 20.0, f'sanity FAILED: L2={_l2:.2f}% (expected ~5.58%)'
print(f'  >> sanity OK')
del _m, _d, _opt, _lbfgs

In [ ]:
# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_one_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    print(f'\n' + '=' * 70)
    print(f'  SEED={seed}  N={N}  width={WIDTH}  domain=({x_lo},{x_hi})')
    print('=' * 70)

    model = PINN(WIDTH).to(device)
    data  = build_data(x_lo, x_hi, T,
                       n_pde, n_data, n_ic, n_bc,
                       speeds, x0s, seed, device)

    print(f'  params={count_params(model):,}  '
          f'PDE={n_pde:,}  Data={n_data:,}  IC={n_ic:,}  BC={n_bc}')

    # ---- Adam ----
    opt_adam = torch.optim.Adam(model.parameters(), lr=ADAM_LR)
    t0 = time.time()
    for i in range(ADAM_ITER):
        opt_adam.zero_grad()
        loss, l_pde, l_ic, l_bc, l_data = compute_losses(model, data)
        loss.backward()
        opt_adam.step()
        if i % 1000 == 0:
            print(f'  Adam  {i:5d} | Total={loss.item():.6e} '
                  f'PDE={l_pde.item():.2e} IC={l_ic.item():.2e} '
                  f'BC={l_bc.item():.2e} Data={l_data.item():.2e}')

    l2_adam   = compute_l2(model, x_lo, x_hi, T, speeds, x0s)
    adam_time = time.time() - t0
    print(f'  >> Adam done {adam_time:.1f}s | L2={l2_adam:.4f}%')

    # ---- L-BFGS ----
    opt_lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=1.0, max_iter=20,
        history_size=50, tolerance_change=1e-12,
        line_search_fn='strong_wolfe'
    )

    t1 = time.time()
    n_steps = LBFGS_ITER // 20   # 100 steps x max_iter=20 = 2000 grad evals
    last_loss = [None]            # cache scalar loss for printing — avoids calling
                                  # compute_losses under no_grad (breaks pde_residual)

    def closure_cached():
        opt_lbfgs.zero_grad()
        l, *_ = compute_losses(model, data)
        l.backward()
        last_loss[0] = l.item()   # .item() extracts scalar before grad graph is freed
        return l

    for j in range(n_steps):
        opt_lbfgs.step(closure_cached)
        if j % 10 == 0:           # print every 200 grad evals
            print(f'  LBFGS  {(j+1)*20:4d} | Total={last_loss[0]:.8e}')

    l2_final   = compute_l2(model, x_lo, x_hi, T, speeds, x0s)
    lbfgs_time = time.time() - t1
    print(f'  >> LBFGS done {lbfgs_time:.1f}s | L2={l2_final:.4f}%')

    # ---- Save checkpoint ----
    ckpt = f'/kaggle/working/checkpoint_N{N}_w{WIDTH}_seed{seed}.pt'
    torch.save({'model_state': model.state_dict(),
                'seed': seed,
                'l2_adam': l2_adam,
                'l2_final': l2_final}, ckpt)
    print(f'  saved {ckpt}')

    # ---- Per-soliton breakdown ----
    print(f'  computing per-soliton L2...')
    ps = per_soliton_l2(model, speeds, x0s, T, device)
    for idx, r in enumerate(ps):
        print(f'    soliton {idx+1:2d}  c={r["c"]:.2f}  x0={r["x0"]:6.1f}  '
              f'local L2 = {r["local_l2_pct"]:7.2f}%')

    ps_l2      = [r['local_l2_pct'] for r in ps]
    worst_idx  = int(np.argmax(ps_l2))
    best_idx   = int(np.argmin(ps_l2))
    print(f'  worst: soliton {worst_idx+1} (c={speeds[worst_idx]:.2f}) '
          f'at {ps_l2[worst_idx]:.2f}%')
    print(f'  best:  soliton {best_idx+1}  (c={speeds[best_idx]:.2f}) '
          f'at {ps_l2[best_idx]:.2f}%')

    return {
        'seed':               seed,
        'l2_adam':            l2_adam,
        'l2_final':           l2_final,
        'adam_time':          adam_time,
        'lbfgs_time':         lbfgs_time,
        'per_soliton':        ps,
        'worst_soliton_idx':  worst_idx + 1,
        'worst_soliton_c':    float(speeds[worst_idx]),
        'worst_soliton_l2':   ps_l2[worst_idx],
        'best_soliton_idx':   best_idx + 1,
        'best_soliton_c':     float(speeds[best_idx]),
        'best_soliton_l2':    ps_l2[best_idx],
    }

In [ ]:
# ============================================================
# MAIN LOOP
# ============================================================

all_results = []

for seed in SEEDS:
    result = train_one_seed(seed)
    all_results.append(result)
    # save after each seed in case run crashes
    with open('/kaggle/working/multiseed_results.json', 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  [running save — {len(all_results)}/{len(SEEDS)} seeds done]')

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================

print('\n' + '=' * 78)
print(f'  SEED SWEEP SUMMARY  —  N={N}  width={WIDTH}')
print('=' * 78)
print(f'{"seed":>6}  {"L2 Adam":>10}  {"L2 Final":>10}  '
      f'{"Worst sol":>10}  {"Worst c":>8}  {"Worst L2%":>10}')
print('-' * 78)
for r in all_results:
    print(f'{r["seed"]:>6}  {r["l2_adam"]:>9.4f}%  {r["l2_final"]:>9.4f}%  '
          f'{r["worst_soliton_idx"]:>10}  {r["worst_soliton_c"]:>8.2f}  '
          f'{r["worst_soliton_l2"]:>9.2f}%')
print('=' * 78)

# reference check on seed 99
r99 = next((r for r in all_results if r['seed'] == 99), None)
if r99:
    print(f'\nSeed 99 final L2 = {r99["l2_final"]:.4f}%  (reference: ~42.15%)')
    status = 'OK' if r99['l2_final'] < 60.0 else 'WARN — check pipeline'
    print(f'  [{status}]')

# key diagnostic
worst_idxs  = [r['worst_soliton_idx'] for r in all_results]
unique_worst = set(worst_idxs)
print(f'\nWorst soliton per seed : {worst_idxs}')
print(f'Unique worst solitons  : {sorted(unique_worst)}')

if len(unique_worst) == 1:
    print('\n=> STRUCTURAL: same soliton fails regardless of seed')
    print('   Failure is config-determined, not optimization-stochastic.')
elif len(unique_worst) <= 3:
    print(f'\n=> PARTIALLY STOCHASTIC: {len(unique_worst)} distinct patterns across {len(SEEDS)} seeds')
    print('   Loss landscape has a small number of dominant partial-basin attractors.')
else:
    print(f'\n=> STOCHASTIC — BROKEN SYMMETRY CONFIRMED')
    print(f'   {len(unique_worst)} distinct failure patterns across {len(SEEDS)} seeds.')
    print('   Loss landscape has multiple partial-basin attractors.')
    print('   Optimizer falls into one stochastically — seed determines which')
    print('   solitons are learned and which are completely missed.')

In [ ]:
# ============================================================
# FIGURE 1: Per-soliton L2 heatmap  (seeds x soliton index)
# Hot = high error. If rows look similar => structural.
# If rows look different => stochastic.
# ============================================================

ps_matrix = np.array([
    [r['local_l2_pct'] for r in res['per_soliton']]
    for res in all_results
])  # shape: (n_seeds, N)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(ps_matrix, aspect='auto', cmap='hot_r',
               vmin=0, vmax=ps_matrix.max())
ax.set_xticks(range(N))
ax.set_xticklabels([f's{i+1}\nc={speeds[i]:.1f}' for i in range(N)], fontsize=8)
ax.set_yticks(range(len(SEEDS)))
ax.set_yticklabels([f'seed={s}' for s in SEEDS], fontsize=9)
ax.set_xlabel('Soliton index (fastest left → slowest right)', fontsize=11)
ax.set_title(
    f'Per-soliton local L² error (%) across 8 seeds — N={N} w={WIDTH}\n'
    f'Similar rows = structural failure | Different rows = stochastic partial-basin',
    fontsize=10
)
plt.colorbar(im, ax=ax, label='Local L² error (%)')
plt.tight_layout()
plt.savefig('/kaggle/working/multiseed_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved multiseed_heatmap.png')

In [ ]:
# ============================================================
# FIGURE 2: Per-soliton bars — all seeds overlaid
# ============================================================

fig, ax = plt.subplots(figsize=(14, 5))
x_pos = np.arange(N)
bar_w = 0.8 / len(SEEDS)
colors = plt.cm.tab10(np.linspace(0, 1, len(SEEDS)))

for k, (res, seed) in enumerate(zip(all_results, SEEDS)):
    ps_l2  = [r['local_l2_pct'] for r in res['per_soliton']]
    offset = (k - len(SEEDS)/2 + 0.5) * bar_w
    ax.bar(x_pos + offset, ps_l2, width=bar_w,
           color=colors[k], alpha=0.85, label=f'seed={seed}')

ax.set_xticks(x_pos)
ax.set_xticklabels([f's{i+1}\nc={speeds[i]:.1f}' for i in range(N)], fontsize=8)
ax.set_ylabel('Local L² error (%)', fontsize=11)
ax.set_xlabel('Soliton index (fastest → slowest)', fontsize=11)
ax.set_title(f'Per-soliton L² error across 8 seeds — N={N} w={WIDTH}', fontsize=12)
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/multiseed_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved multiseed_bars.png')

In [ ]:
# ============================================================
# FIGURE 3: Global L2 variance across seeds (Adam vs Final)
# ============================================================

final_l2s = [r['l2_final'] for r in all_results]
adam_l2s  = [r['l2_adam']  for r in all_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vals, phase, col in zip(
        axes,
        [adam_l2s, final_l2s],
        ['After Adam (15k iters)', 'After L-BFGS (2k iters)'],
        ['steelblue', 'darkorange']):
    ax.bar(range(len(SEEDS)), vals, color=col, alpha=0.85)
    ax.set_xticks(range(len(SEEDS)))
    ax.set_xticklabels([str(s) for s in SEEDS])
    ax.set_xlabel('Seed', fontsize=11)
    ax.set_ylabel('L² error (%)', fontsize=11)
    ax.set_title(phase, fontsize=11)
    ax.axhline(np.mean(vals), color='red', ls='--',
               label=f'mean={np.mean(vals):.1f}%  std={np.std(vals):.1f}%')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(f'N={N} w={WIDTH} — global L² variance across 8 seeds', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/multiseed_global_l2.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved multiseed_global_l2.png')

print(f'\nGlobal L2 stats:')
print(f'  After Adam:  mean={np.mean(adam_l2s):.2f}%  std={np.std(adam_l2s):.2f}%  '
      f'min={np.min(adam_l2s):.2f}%  max={np.max(adam_l2s):.2f}%')
print(f'  After LBFGS: mean={np.mean(final_l2s):.2f}%  std={np.std(final_l2s):.2f}%  '
      f'min={np.min(final_l2s):.2f}%  max={np.max(final_l2s):.2f}%')

In [ ]:
# ============================================================
# FIGURE 4: Per-soliton mean +/- std across seeds
# High std on a soliton = stochastic (which basin it lands in matters)
# Low std = structural (always fails/succeeds regardless of seed)
# ============================================================

per_sol_mean = ps_matrix.mean(axis=0)
per_sol_std  = ps_matrix.std(axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(N), per_sol_mean, yerr=per_sol_std,
       capsize=4, color='steelblue', alpha=0.85, ecolor='black')
ax.set_xticks(range(N))
ax.set_xticklabels([f's{i+1}\nc={speeds[i]:.1f}' for i in range(N)], fontsize=8)
ax.set_ylabel('Local L² error (%)\nmean ± std across 8 seeds', fontsize=10)
ax.set_xlabel('Soliton index (fastest → slowest)', fontsize=11)
ax.set_title(
    f'Per-soliton mean±std across 8 seeds — N={N} w={WIDTH}\n'
    f'Large std = seed-sensitive (stochastic)  |  Small std = seed-insensitive (structural)',
    fontsize=10
)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/multiseed_mean_std.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved multiseed_mean_std.png')

print('\nPer-soliton mean ± std:')
for i in range(N):
    print(f'  s{i+1:2d} (c={speeds[i]:.1f}): '
          f'mean={per_sol_mean[i]:6.1f}%  std={per_sol_std[i]:5.1f}%')
print(f'\nMost variable : s{np.argmax(per_sol_std)+1} '
      f'(c={speeds[np.argmax(per_sol_std)]:.1f})  std={per_sol_std.max():.1f}%')
print(f'Least variable: s{np.argmin(per_sol_std)+1} '
      f'(c={speeds[np.argmin(per_sol_std)]:.1f})  std={per_sol_std.min():.1f}%')

In [ ]:
# ============================================================
# SAVE CSV
# ============================================================

rows = []
for res in all_results:
    row = {
        'seed':              res['seed'],
        'l2_adam':           res['l2_adam'],
        'l2_final':          res['l2_final'],
        'adam_time_s':       res['adam_time'],
        'lbfgs_time_s':      res['lbfgs_time'],
        'worst_soliton_idx': res['worst_soliton_idx'],
        'worst_soliton_c':   res['worst_soliton_c'],
        'worst_soliton_l2':  res['worst_soliton_l2'],
        'best_soliton_idx':  res['best_soliton_idx'],
        'best_soliton_c':    res['best_soliton_c'],
        'best_soliton_l2':   res['best_soliton_l2'],
    }
    for i, ps in enumerate(res['per_soliton']):
        row[f'sol{i+1}_l2'] = ps['local_l2_pct']
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv('/kaggle/working/multiseed_results.csv', index=False)
print('saved multiseed_results.csv')
print()
print(df[['seed','l2_adam','l2_final',
          'worst_soliton_idx','worst_soliton_c','worst_soliton_l2']].to_string(index=False))

In [ ]:
# ============================================================
# FINAL INTERPRETATION
# ============================================================

print('\n' + '=' * 60)
print('FINAL INTERPRETATION')
print('=' * 60)

worst_idxs   = [r['worst_soliton_idx'] for r in all_results]
unique_worst  = set(worst_idxs)
n_unique      = len(unique_worst)

print(f'Worst soliton per seed: {worst_idxs}')
print(f'Unique:                 {sorted(unique_worst)}')
print(f'N seeds:                {len(SEEDS)}')
print(f'N unique patterns:      {n_unique}')

if n_unique == 1:
    sol = list(unique_worst)[0]
    print(f'\n=> STRUCTURAL FAILURE')
    print(f'   Soliton {sol} (c={speeds[sol-1]:.1f}) fails across all {len(SEEDS)} seeds.')
    print(f'   The partial-basin failure is config-determined, not seed-sensitive.')
    print(f'   Likely cause: that soliton occupies a structurally harder region')
    print(f'   (speed-crowding between s{sol-1} and s{sol+1}) that any init struggles with.')
    print(f'   Implication for paper: failure is predictable from the config alone.')
elif n_unique <= 3:
    print(f'\n=> PARTIALLY STOCHASTIC')
    print(f'   {n_unique} distinct failure patterns across {len(SEEDS)} seeds.')
    print(f'   Small number of dominant partial-basin attractors in loss landscape.')
    print(f'   Implication: optimizer has a few bad basins it tends to fall into.')
else:
    print(f'\n=> STOCHASTIC — BROKEN SYMMETRY CONFIRMED')
    print(f'   {n_unique} distinct failure patterns across {len(SEEDS)} seeds.')
    print(f'   Loss landscape has many partial-basin attractors.')
    print(f'   Which solitons get learned is determined by the random seed, not the config.')
    print(f'   This is the broken-symmetry regime: multiple near-equivalent bad solutions.')
    print(f'   Implication: data anchoring (n_data=2000) should select the correct basin')
    print(f'   and collapse variance across seeds — a clean follow-up experiment.')

print('\nDone. All outputs in /kaggle/working')
print('  multiseed_results.json')
print('  multiseed_results.csv')
print('  multiseed_heatmap.png')
print('  multiseed_bars.png')
print('  multiseed_global_l2.png')
print('  multiseed_mean_std.png')
print(f'  checkpoint_N12_w50_seed{{seed}}.pt  (one per seed)')